# 4a · Chunk & embed the manuals

**Core · Live-code · ~20 min**

To let ARIA answer from the manuals, we first have to make them **searchable by meaning**.
That's a three-part job: split each manual into focused pieces (**chunking**), turn each
piece into a vector (**embedding**, from Module 2), and store those vectors somewhere we can
search quickly (a **vector store**). This notebook does all three.

Solution: [`solution/04a_embed_docs.ipynb`](solution/04a_embed_docs.ipynb).

In [ ]:
import sys, os, re
from pathlib import Path
sys.path.insert(0, os.path.abspath(".."))
from shared.llm import embed

# Find all five manual files.
manuals = sorted(Path("../data/manuals").glob("*.md"))
print("Manuals found:", [m.name for m in manuals])

## Step 1 — Chunking: split each manual into sections

Why not embed a whole manual as one vector? Because a single vector for a long, multi-topic
document is *blurry* — it averages everything together, so it matches nothing well. Instead we
split each manual at its section headings, giving one crisp chunk per procedure. When ARIA
later asks about oxygen, we can return *just* the oxygen section.

The helper below uses a regular expression to split the text right before each markdown
heading (`#`, `##`, or `###`), and drops any tiny fragments. Each chunk remembers which file
it came from so we can cite it later.

In [ ]:
def chunk_markdown(text, source):
    # Split the document just before every markdown heading line.
    parts = re.split(r"\n(?=#{1,3}\s)", text)
    # Keep the non-trivial sections, tagging each with its source filename.
    return [{"text": p.strip(), "source": source} for p in parts if len(p.strip()) >= 40]

# TODO: build one big list called `chunks` by chunking every manual.
chunks = []
#   for m in manuals:
#       chunks += chunk_markdown(m.read_text(), m.name)

print(f"{len(chunks)} chunks total")
if chunks:
    print("Example chunk:\n", chunks[0]["text"][:200])

## Step 2 — Embed the chunks and store them in Chroma

Now we turn each chunk's text into a vector and load it into **Chroma**, a lightweight local
vector database. A few notes:

- We compute the embeddings ourselves with Ollama and pass them in (`embeddings=...`). This
  keeps everything offline — Chroma won't try to download its own model.
- Each item needs a unique **id**. We also store the original **document** text and its
  **metadata** (the source filename) so retrieval can return readable text with a citation.
- `EphemeralClient()` keeps the database in memory for this session — perfect for a workshop.

In [ ]:
import chromadb

# TODO 1: embed every chunk's text into a list of vectors.
#   vectors = embed([c["text"] for c in chunks])

# Create an in-memory Chroma collection to hold our manual chunks.
client = chromadb.EphemeralClient()
collection = client.create_collection("orbital_manuals")

# TODO 2: add the chunks to the collection. Fill in embeddings=vectors below.
#   collection.add(
#       ids=[f"c{i}" for i in range(len(chunks))],
#       embeddings=vectors,
#       documents=[c["text"] for c in chunks],
#       metadatas=[{"source": c["source"]} for c in chunks],
#   )

print("chunks stored in the vector database:", collection.count())

## ✅ Recap

You split the manuals into focused chunks, embedded each one, and loaded them into a
searchable vector store. That store is now ARIA's library. In **4b** we'll query it: given a
question, find the closest chunks and have ARIA answer from them.

> The exact same steps are wrapped up in `shared/rag.py` as `RagIndex.from_manuals()` — we'll
> use that convenience version in Modules 5 and 7.

# 4a · Chunk & embed the manuals

**Core · Live-code · ~20 min**

Turn the five Orbital manuals into a searchable vector store with Chroma.

> Needs Ollama with `nomic-embed-text`. Solution: [`solution/04a_embed_docs.ipynb`](solution/04a_embed_docs.ipynb).

In [ ]:
import sys, os, re
from pathlib import Path
sys.path.insert(0, os.path.abspath(".."))
from shared.llm import embed

manuals = sorted(Path("../data/manuals").glob("*.md"))
print([m.name for m in manuals])

## Chunking

LLMs work best with focused context. Instead of dumping a whole manual, we split each
document into **sections** (by `##` headings) so we can retrieve just the relevant part.

In [ ]:
def chunk_markdown(text, source):
    parts = re.split(r"\n(?=#{1,3}\s)", text)   # split before headings
    return [{"text": p.strip(), "source": source} for p in parts if len(p.strip()) >= 40]

# TODO: build a list `chunks` by chunking every manual
chunks = []
# for m in manuals:
#     chunks += chunk_markdown(m.read_text(), m.name)
print(f"{len(chunks)} chunks")

## Embed & store in Chroma

We compute embeddings ourselves (with Ollama) and hand them to Chroma, so nothing needs
to download.

In [ ]:
import chromadb

# TODO: embed every chunk's text -> `vectors`
# vectors = embed([c["text"] for c in chunks])

client = chromadb.EphemeralClient()
collection = client.create_collection("orbital_manuals")
# TODO: add ids, embeddings=vectors, documents, and metadatas (source) to the collection
# collection.add(ids=[f"c{i}" for i in range(len(chunks))], embeddings=vectors,
#                documents=[c["text"] for c in chunks],
#                metadatas=[{"source": c["source"]} for c in chunks])
print("stored:", collection.count())

In **4b** we'll query this collection to answer questions. (There's also a ready-made
`shared/rag.py` that wraps all of this — we'll use it in Modules 5 and 7.)